# XGBoost Regression

training xgboost regression in blocks using baseline features + traficom features


## 1. Imports

This notebook keeps the same overall workflow as the linear-regression and random-forest notebooks, but uses native-categorical XGBoost with validation-based early stopping and a compact tuning block across stable configurations, target modes, and feature variants.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


## 2. Load data

In [244]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [245]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

for dataset_df in [train_df, validation_df]:
    for date_column in ["first_seen_date", "last_seen_date", "scrape_date"]:
        if date_column in dataset_df.columns:
            dataset_df[date_column] = pd.to_datetime(dataset_df[date_column], errors="coerce")

reference_first_seen_date = min(
    df["first_seen_date"].min()
    for df in [train_df, validation_df]
    if "first_seen_date" in df.columns
)

for dataset_df in [train_df, validation_df]:
    dataset_df["first_seen_day_offset"] = (
        dataset_df["first_seen_date"] - reference_first_seen_date
    ).dt.days
    dataset_df["last_seen_day_offset"] = (
        dataset_df["last_seen_date"] - reference_first_seen_date
    ).dt.days
    dataset_df["listing_midpoint_day_offset"] = (
        dataset_df["first_seen_day_offset"]
        + dataset_df["last_seen_day_offset"]
    ) / 2


## 3. Quick data checks

In [246]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (7954, 88)
Validation shape: (1689, 88)


In [247]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

Train info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7954 entries, 0 to 7953
Data columns (total 88 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   product_id                        7954 non-null   int64         
 1   part_name                         7954 non-null   object        
 2   price                             7954 non-null   float64       
 3   quality_grade                     7954 non-null   object        
 4   oem_number                        7954 non-null   object        
 5   mileage                           7954 non-null   float64       
 6   brand                             7954 non-null   object        
 7   model                             7954 non-null   object        
 8   category                          7954 non-null   object        
 9   subcategory                       7954 non-null   object        
 10  scrape_date                       795

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [248]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates
    if column in train_df.columns
]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates
    if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

listing_date_offset_features = [
    "first_seen_day_offset",
    "last_seen_day_offset",
    "listing_midpoint_day_offset",
]

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
    *listing_date_offset_features,
]

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

recommended_inclusion_reasons = {
    "first_seen_day_offset": "Numeric position of listing start within the crawl window for XGBoost.",
    "last_seen_day_offset": "Numeric position of listing end within the crawl window for XGBoost.",
    "listing_midpoint_day_offset": "Numeric midpoint of listing visibility window for XGBoost.",
    "brand_is_known_model_family": "Low-cardinality metadata flag; slight validation improvement when included.",
    "mileage_missing_flag": "Tracks rows where mileage is missing after cleanup before model-time imputation or native missing handling.",
}

recommended_exclusion_reasons = {
    "product_id": "High-cardinality listing identifier; encourages memorization and hurt MAE.",
    "scrape_date": "Current crawl wave rather than part value; worsened validation when included.",
    "brand_merge_key": "Redundant normalized brand key that overlaps with brand.",
    "model_merge_key": "Redundant normalized model key that overlaps with model.",
    "model_family_clean": "Collapsed model family duplicate of the existing model labels.",
    "model_looks_like_part_taxonomy": "Constant in the training split, so it adds no signal.",
}

manual_all_feature_groups = list(dict.fromkeys(
    baseline_features
    + traficom_features
    + registry_lifecycle_features
    + traficom_extended_features
    + listing_dynamics_features
))

xgboost_specific_exclusion_reasons = {
    "first_seen_date": "Raw date string replaced with numeric day-offset feature for XGBoost.",
    "last_seen_date": "Raw date string replaced with numeric day-offset feature for XGBoost.",
}

xgboost_leakage_risk_feature_reasons = {
    "times_observed": "Uses the full number of listing observations across the crawl window, including future rows.",
    "observed_span_days": "Uses the final listing lifetime, which depends on future observations.",
    "price_changed_flag": "Uses whether the price ever changed over the full listing history.",
    "price_change_count": "Uses the total number of price changes over the full listing history.",
    "absolute_price_change": "Uses the difference between last observed and first observed price.",
    "relative_price_change_pct": "Uses the full-history price change percentage from first to last observation.",
    "price_changed_flag_so_far": "Uses the current row's price to decide whether the listing has changed price so far, so it leaks target information.",
    "price_change_count_so_far": "Counts price changes including the current row's price, so it leaks target information.",
    "absolute_price_change_so_far": "Computed from the current row's price minus the first observed price, so it directly encodes the target.",
    "relative_price_change_pct_so_far": "Computed from the current row's price relative to the first observed price, so it directly encodes the target.",
    "last_seen_day_offset": "Uses the final observed date of the listing, which is future information for earlier rows.",
    "listing_midpoint_day_offset": "Uses the full listing span and therefore depends on the final observed date.",
}

recommended_excluded_features = set(recommended_exclusion_reasons)
xgboost_excluded_features = recommended_excluded_features | set(xgboost_specific_exclusion_reasons)
xgboost_leakage_risk_features = set(xgboost_leakage_risk_feature_reasons)
xgboost_trusted_excluded_features = xgboost_excluded_features | xgboost_leakage_risk_features

recommended_model_features = [
    column for column in train_df.columns
    if column != "price" and column not in xgboost_excluded_features
]

recommended_model_features_without_date_offsets = [
    column for column in recommended_model_features
    if column not in listing_date_offset_features
]

manual_all_xgboost_features = [
    column for column in manual_all_feature_groups
    if column not in xgboost_excluded_features
]

recommended_model_features_leakage_safe = [
    column for column in train_df.columns
    if column != "price" and column not in xgboost_trusted_excluded_features
]

recommended_model_features_leakage_safe_without_date_offsets = [
    column for column in recommended_model_features_leakage_safe
    if column not in listing_date_offset_features
]

manual_all_xgboost_features_leakage_safe = [
    column for column in manual_all_xgboost_features
    if column not in xgboost_leakage_risk_features
]

missing_from_previous_manual_all = [
    column for column in recommended_model_features if column not in manual_all_feature_groups
]

feature_audit_df = pd.DataFrame(
    [
        {
            "feature": column,
            "status": "missing_from_previous_manual_all",
            "reason": recommended_inclusion_reasons.get(
                column,
                "New candidate feature found in the dataset and included in the recommended model feature set.",
            ),
        }
        for column in missing_from_previous_manual_all
    ]
    + [
        {
            "feature": column,
            "status": "recommended_exclusion",
            "reason": recommended_exclusion_reasons[column],
        }
        for column in sorted(recommended_excluded_features)
    ]
    + [
        {
            "feature": column,
            "status": "xgboost_specific_exclusion",
            "reason": xgboost_specific_exclusion_reasons[column],
        }
        for column in sorted(xgboost_specific_exclusion_reasons)
    ]
    + [
        {
            "feature": column,
            "status": "leakage_risk_exclusion",
            "reason": xgboost_leakage_risk_feature_reasons[column],
        }
        for column in sorted(xgboost_leakage_risk_features)
    ]
)

print("Columns missing from the previous manual all-features set:")
print(missing_from_previous_manual_all)

print("\nRecommended exclusions:")
print(sorted(recommended_excluded_features))

print("\nXGBoost-specific exclusions:")
print(sorted(xgboost_specific_exclusion_reasons))

print("\nLeakage-risk exclusions for trusted model selection:")
print(sorted(xgboost_leakage_risk_features))

print("\nNumber of exploratory recommended model features:", len(recommended_model_features))
print("Number of trusted recommended model features:", len(recommended_model_features_leakage_safe))
print("Number of trusted recommended model features without date offsets:", len(recommended_model_features_leakage_safe_without_date_offsets))
print("Number of trusted manual all-features usable by XGBoost:", len(manual_all_xgboost_features_leakage_safe))

feature_audit_df

Found registry lifecycle features:
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']

Missing registry lifecycle features:
[]

Found extended Traficom features:
['model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over_10y', 'brand_share_over_200k_km', 'brand_automatic_share', 'brand_petrol_share', 'brand_diesel_share', 'brand_ev_s

,feature,status,reason
0,brand_is_known_model_family,missing_from_previous_manual_all,Low-cardinality metadata flag; slight validati...
1,mileage_missing_flag,missing_from_previous_manual_all,Tracks rows where mileage was originally missi...
2,observations_so_far,missing_from_previous_manual_all,New candidate feature found in the dataset and...
3,days_since_first_seen_so_far,missing_from_previous_manual_all,New candidate feature found in the dataset and...
4,price_changed_flag_so_far,missing_from_previous_manual_all,New candidate feature found in the dataset and...
5,price_change_count_so_far,missing_from_previous_manual_all,New candidate feature found in the dataset and...
6,absolute_price_change_so_far,missing_from_previous_manual_all,New candidate feature found in the dataset and...
7,relative_price_change_pct_so_far,missing_from_previous_manual_all,New candidate feature found in the dataset and...
8,brand_merge_key,recommended_exclusion,Redundant normalized brand key that overlaps w...
9,model_family_clean,recommended_exclusion,Collapsed model family duplicate of the existi...


## 5. Select baseline feature set

In [249]:
features_baseline_only = list(dict.fromkeys(
    [
        feature for feature in baseline_features
        if feature not in xgboost_trusted_excluded_features
    ]
    + [
        "first_seen_day_offset",
    ]
))

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

Number of baseline features: 14


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'first_seen_day_offset']

## 6. Build X and y

In [250]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [251]:
y_train_log = np.log(y_train)
y_validation_log = np.log(y_validation)

y_train_log.head()


0    5.179534
1    5.179534
2    5.179534
3    5.179534
4    3.165475
Name: price, dtype: float64

## 8. Detect column types

In [252]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features

['mileage',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'first_seen_day_offset']

## 8. Detect column types

In [253]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [254]:
if "numeric_features" not in globals() or "categorical_features" not in globals():
    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

column_type_summary = pd.DataFrame(
    {
        "column_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
        "columns": [numeric_features, categorical_features],
    }
)

display(column_type_summary)

,column_type,count,columns
0,numeric,6,"[mileage, year_start, year_end, year_span, yea..."
1,categorical,8,"[part_name, quality_grade, oem_number, brand, ..."


## 9. Tune Native-Categorical XGBoost

This tuning block now focuses only on the strongest trusted feature family so it finishes much faster. The broader cross-feature comparison still happens later in the notebook.

In [255]:
xgboost_candidate_configs = {
    "log_absoluteerror_balanced": {
        "target_mode": "log",
        "model_params": {
            "objective": "reg:absoluteerror",
            "eval_metric": "mae",
            "n_estimators": 2200,
            "learning_rate": 0.025,
            "max_depth": 5,
            "min_child_weight": 5,
            "gamma": 0.05,
            "subsample": 0.80,
            "colsample_bytree": 0.70,
            "colsample_bylevel": 0.80,
            "reg_alpha": 0.20,
            "reg_lambda": 3.00,
            "max_bin": 256,
            "max_cat_to_onehot": 10,
            "max_cat_threshold": 64,
        },
        "notes": "Direct MAE optimization with broader categorical one-hot handling.",
    },
    "log_absoluteerror_regularized": {
        "target_mode": "log",
        "model_params": {
            "objective": "reg:absoluteerror",
            "eval_metric": "mae",
            "n_estimators": 2600,
            "learning_rate": 0.020,
            "max_depth": 4,
            "min_child_weight": 8,
            "gamma": 0.10,
            "subsample": 0.75,
            "colsample_bytree": 0.65,
            "colsample_bylevel": 0.80,
            "reg_alpha": 0.40,
            "reg_lambda": 4.50,
            "max_bin": 256,
            "max_cat_to_onehot": 10,
            "max_cat_threshold": 64,
        },
        "notes": "More regularized MAE-focused variant for stability.",
    },
    "log_sqerror_balanced": {
        "target_mode": "log",
        "model_params": {
            "objective": "reg:squarederror",
            "eval_metric": "mae",
            "n_estimators": 1800,
            "learning_rate": 0.030,
            "max_depth": 5,
            "min_child_weight": 5,
            "gamma": 0.08,
            "subsample": 0.80,
            "colsample_bytree": 0.70,
            "colsample_bylevel": 0.80,
            "reg_alpha": 0.20,
            "reg_lambda": 3.25,
            "max_bin": 256,
            "max_cat_to_onehot": 10,
            "max_cat_threshold": 64,
        },
        "notes": "Squared-error reference aligned more closely with the trusted categorical setup.",
    },
    "raw_absoluteerror_balanced": {
        "target_mode": "raw",
        "model_params": {
            "objective": "reg:absoluteerror",
            "eval_metric": "mae",
            "n_estimators": 1800,
            "learning_rate": 0.025,
            "max_depth": 5,
            "min_child_weight": 5,
            "gamma": 0.05,
            "subsample": 0.80,
            "colsample_bytree": 0.70,
            "colsample_bylevel": 0.80,
            "reg_alpha": 0.20,
            "reg_lambda": 3.00,
            "max_bin": 256,
            "max_cat_to_onehot": 10,
            "max_cat_threshold": 64,
        },
        "notes": "Raw-target MAE optimization check on the same categorical settings.",
    },
    "raw_sqerror_reference": {
        "target_mode": "raw",
        "model_params": {
            "objective": "reg:squarederror",
            "eval_metric": "mae",
            "n_estimators": 1800,
            "learning_rate": 0.030,
            "max_depth": 5,
            "min_child_weight": 5,
            "gamma": 0.08,
            "subsample": 0.80,
            "colsample_bytree": 0.70,
            "colsample_bylevel": 0.80,
            "reg_alpha": 0.20,
            "reg_lambda": 3.25,
            "max_bin": 256,
            "max_cat_to_onehot": 10,
            "max_cat_threshold": 64,
        },
        "notes": "Raw-target squarederror reference with lower child-weight and broader categorical one-hot handling.",
    },
}


def align_xgboost_frames(X_train_current, X_validation_current):
    X_train_prepared_current = X_train_current.copy()
    X_validation_prepared_current = X_validation_current.copy()

    datetime_columns = X_train_prepared_current.select_dtypes(
        include=["datetime64[ns]", "datetimetz"]
    ).columns.tolist()
    if len(datetime_columns) > 0:
        X_train_prepared_current = X_train_prepared_current.drop(columns=datetime_columns)
        X_validation_prepared_current = X_validation_prepared_current.drop(
            columns=datetime_columns,
            errors="ignore",
        )

    bool_columns = X_train_prepared_current.select_dtypes(include=["bool"]).columns.tolist()
    for column in bool_columns:
        X_train_prepared_current[column] = X_train_prepared_current[column].astype(int)
        X_validation_prepared_current[column] = X_validation_prepared_current[column].astype(int)

    categorical_columns = X_train_prepared_current.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
    for column in categorical_columns:
        train_as_string = X_train_prepared_current[column].astype("string")
        validation_as_string = X_validation_prepared_current[column].astype("string")

        categories = pd.Index(train_as_string.dropna().unique())
        if len(categories) == 0:
            categories = pd.Index(["__missing__"])

        X_train_prepared_current[column] = pd.Categorical(
            train_as_string,
            categories=categories,
        )
        X_validation_prepared_current[column] = pd.Categorical(
            validation_as_string,
            categories=categories,
        )

    return X_train_prepared_current, X_validation_prepared_current


def make_xgboost_regressor(params=None):
    model_params = {
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 42,
        "n_jobs": -1,
        "early_stopping_rounds": 120,
    }
    if params is not None:
        model_params.update(params)

    return XGBRegressor(**model_params)


def prepare_xgboost_target(y_series, target_mode):
    if target_mode == "log":
        return np.log(y_series)
    if target_mode == "raw":
        return y_series.copy()

    raise ValueError(f"Unsupported target_mode: {target_mode}")


def convert_xgboost_predictions_to_eur(predictions, target_mode, y_train_reference=None):
    predictions = np.asarray(predictions, dtype=float)

    if target_mode == "log":
        upper_price_bound = float(y_train_reference.max()) * 10 if y_train_reference is not None else 1_000_000.0
        clipped_predictions = np.nan_to_num(
            predictions,
            nan=0.0,
            posinf=np.log(upper_price_bound),
            neginf=np.log(1e-3),
        )
        clipped_predictions = np.clip(
            clipped_predictions,
            a_min=np.log(1e-3),
            a_max=np.log(upper_price_bound),
        )
        return np.exp(clipped_predictions)

    if target_mode == "raw":
        return np.clip(
            np.nan_to_num(predictions, nan=0.0, posinf=0.0, neginf=0.0),
            a_min=0.0,
            a_max=None,
        )

    raise ValueError(f"Unsupported target_mode: {target_mode}")


def fit_xgboost_model(
    X_train_current,
    y_train_current,
    X_validation_current,
    y_validation_current,
    params,
):
    X_train_prepared_current, X_validation_prepared_current = align_xgboost_frames(
        X_train_current,
        X_validation_current,
    )

    model_current = make_xgboost_regressor(params)
    model_current.fit(
        X_train_prepared_current,
        y_train_current,
        eval_set=[(X_validation_prepared_current, y_validation_current)],
        verbose=False,
    )

    return model_current, X_train_prepared_current, X_validation_prepared_current


In [ ]:
xgboost_tuning_feature_set_candidates = {
    "trusted_registry_lifecycle_stack": {
        "features": (
            baseline_features
            + traficom_features
            + registry_lifecycle_features
        ),
        "trusted_for_selection": True,
        "priority": 1,
    },
    "trusted_registry_lifecycle_stack_without_oem_number": {
        "features": [
            column for column in (
                baseline_features
                + traficom_features
                + registry_lifecycle_features
            )
            if column != "oem_number"
        ],
        "trusted_for_selection": True,
        "priority": 2,
    },
    "trusted_recommended_features_without_date_offsets": {
        "features": recommended_model_features_leakage_safe_without_date_offsets,
        "trusted_for_selection": True,
        "priority": 3,
    },
    "trusted_recommended_features_without_date_offsets_without_oem_number": {
        "features": [
            column for column in recommended_model_features_leakage_safe_without_date_offsets
            if column != "oem_number"
        ],
        "trusted_for_selection": True,
        "priority": 4,
    },
}

xgboost_tuning_feature_sets = {}
seen_tuning_feature_signatures = set()
for feature_variant_name, feature_variant_config in xgboost_tuning_feature_set_candidates.items():
    available_tuning_features = [
        column for column in dict.fromkeys(feature_variant_config["features"])
        if column in train_df.columns
    ]
    feature_signature = tuple(available_tuning_features)
    if len(available_tuning_features) == 0 or feature_signature in seen_tuning_feature_signatures:
        continue

    seen_tuning_feature_signatures.add(feature_signature)
    xgboost_tuning_feature_sets[feature_variant_name] = {
        **feature_variant_config,
        "features": available_tuning_features,
    }

if len(xgboost_tuning_feature_sets) == 0:
    raise NameError("No valid XGBoost tuning feature sets were available in train_df.")

tuning_results = []
for feature_variant_name, feature_variant_config in xgboost_tuning_feature_sets.items():
    tuning_features = feature_variant_config["features"]
    X_train_tuning = train_df[tuning_features].copy()
    X_validation_tuning = validation_df[tuning_features].copy()

    for config_name, config in xgboost_candidate_configs.items():
        y_train_tuning = prepare_xgboost_target(y_train, config["target_mode"])
        y_validation_tuning = prepare_xgboost_target(y_validation, config["target_mode"])

        model_current, X_train_tuning_prepared, X_validation_tuning_prepared = fit_xgboost_model(
            X_train_tuning,
            y_train_tuning,
            X_validation_tuning,
            y_validation_tuning,
            config["model_params"],
        )

        validation_pred_model_scale_current = model_current.predict(X_validation_tuning_prepared)
        validation_pred_current = convert_xgboost_predictions_to_eur(
            validation_pred_model_scale_current,
            config["target_mode"],
            y_train_reference=y_train,
        )

        tuning_results.append({
            "config": config_name,
            "feature_variant": feature_variant_name,
            "feature_variant_priority": feature_variant_config["priority"],
            "trusted_for_selection": feature_variant_config["trusted_for_selection"],
            "raw_columns_used": len(tuning_features),
            "target_mode": config["target_mode"],
            "validation_MAE": mean_absolute_error(y_validation, validation_pred_current),
            "validation_RMSE": np.sqrt(mean_squared_error(y_validation, validation_pred_current)),
            "validation_R2": r2_score(y_validation, validation_pred_current),
            "best_iteration": getattr(model_current, "best_iteration", None),
            "notes": config["notes"],
        })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    ["trusted_for_selection", "validation_MAE", "validation_RMSE", "feature_variant_priority"],
    ascending=[False, True, True, True],
).reset_index(drop=True)

trusted_tuning_results_df = tuning_results_df[
    tuning_results_df["trusted_for_selection"]
].copy()
if trusted_tuning_results_df.empty:
    trusted_tuning_results_df = tuning_results_df.copy()

# Local notebook selection stays on the fixed train/validation split.
# GroupKFold reranking is reserved for the Puhti tuning scripts.
selected_validation_row = trusted_tuning_results_df.sort_values(
    ["validation_MAE", "validation_RMSE", "feature_variant_priority"],
    ascending=[True, True, True],
).iloc[0]
selected_xgboost_config_name = selected_validation_row["config"]
selected_xgboost_feature_variant_name = selected_validation_row["feature_variant"]
selected_xgboost_config = xgboost_candidate_configs[selected_xgboost_config_name]
selected_xgboost_params = selected_xgboost_config["model_params"]

print("Selected trusted XGBoost config on fixed validation split:", selected_xgboost_config_name)
print("Selected trusted XGBoost tuning feature variant:", selected_xgboost_feature_variant_name)
print("GroupKFold reranking is run only in the Puhti tuning scripts.")
print(selected_xgboost_config)

print("Validation search results:")
display(
    tuning_results_df.style.format({
        "validation_MAE": "{:.4f}",
        "validation_RMSE": "{:.4f}",
        "validation_R2": "{:.4f}",
    })
)


## 10. Train tuned baseline XGBoost


In [257]:
baseline_model, X_train_prepared, X_validation_prepared = fit_xgboost_model(
    X_train,
    prepare_xgboost_target(y_train, selected_xgboost_config["target_mode"]),
    X_validation,
    prepare_xgboost_target(y_validation, selected_xgboost_config["target_mode"]),
    selected_xgboost_params,
)

## 11. Predict and convert back to euro scale

In [258]:
validation_pred_model_scale = baseline_model.predict(X_validation_prepared)

In [259]:
validation_pred = convert_xgboost_predictions_to_eur(
    validation_pred_model_scale,
    selected_xgboost_config["target_mode"],
    y_train_reference=y_train,
)

## 12. Preview Baseline XGBoost Predictions


In [260]:
baseline_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred,
})

baseline_prediction_preview.head()

,actual_price,predicted_price
0,177.6,182.371368
1,473.6,437.427521
2,142.1,141.253571
3,82.9,90.480766
4,177.6,115.932884


## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [261]:
features_baseline_plus_traficom = list(dict.fromkeys(
    features_baseline_only + traficom_features
))

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

Number of baseline + Traficom features: 27


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'first_seen_day_offset',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage']

## 14. Build X and y for baseline + Traficom

In [262]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [263]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [264]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'first_seen_day_offset', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 16. Prepare and train baseline + Traficom XGBoost


In [265]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()


In [266]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

xgboost_traficom, X_train_traficom_prepared, X_validation_traficom_prepared = fit_xgboost_model(
    X_train_traficom,
    prepare_xgboost_target(y_train, selected_xgboost_config["target_mode"]),
    X_validation_traficom,
    prepare_xgboost_target(y_validation, selected_xgboost_config["target_mode"]),
    selected_xgboost_params,
)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'first_seen_day_offset', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 17. Predict baseline + Traficom XGBoost


In [267]:
validation_pred_model_scale_traficom = xgboost_traficom.predict(X_validation_traficom_prepared)

## 18. Predict on validation and convert back to euro scale

In [268]:
validation_pred_traficom = convert_xgboost_predictions_to_eur(
    validation_pred_model_scale_traficom,
    selected_xgboost_config["target_mode"],
    y_train_reference=y_train,
)

In [269]:
validation_pred_traficom

array([180.77713013, 429.56420898, 141.68617249, ..., 215.95600891,
       233.80265808, 232.921875  ], shape=(1689,))

## 19. Preview Baseline + Traficom XGBoost Predictions


In [270]:
traficom_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_traficom,
})

traficom_prediction_preview.head()

,actual_price,predicted_price
0,177.6,180.777130
1,473.6,429.564209
2,142.1,141.686172
3,82.9,93.514381
4,177.6,123.758598


## 20. All Recommended Features

This section trains the XGBoost model on every recommended feature: the full manual feature union plus `first_seen_date`, `last_seen_date`, and `brand_is_known_model_family`, while still excluding IDs, duplicate keys, and the constant taxonomy flag.


In [271]:
features_all = recommended_model_features_leakage_safe

print("Number of trusted recommended model features:", len(features_all))
features_all

Number of trusted recommended model features: 67


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'brand_is_known_model_family',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'model_share_of_market',
 'model_share_within_brand',
 'model_share_over_10y',
 'model_share_over_200k_km',
 'model_automatic_share',
 'model_petrol_share',
 'model_diesel_share',
 'model_ev_share',
 'model_hybrid_share',
 'model_turbo_share',
 'model_firstreg_total_2014_2026',
 'model_firstreg_year_span',
 'model_firstreg_peak_year',
 'model_firstreg_peak_count',
 'model_firstreg_recent_share',
 'model_firstreg_old_share',
 'model_firstreg_weighted_year',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_

## 21. Build X and y for all recommended features

In [272]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [273]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [274]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 23. Prepare and train recommended-feature XGBoost


In [275]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()


In [276]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

xgboost_all, X_train_all_prepared, X_validation_all_prepared = fit_xgboost_model(
    X_train_all,
    prepare_xgboost_target(y_train, selected_xgboost_config["target_mode"]),
    X_validation_all,
    prepare_xgboost_target(y_validation, selected_xgboost_config["target_mode"]),
    selected_xgboost_params,
)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 24. Predict recommended-feature XGBoost


In [277]:
validation_pred_model_scale_all = xgboost_all.predict(X_validation_all_prepared)

## 25. Predict on validation and convert back to euro scale

In [278]:
validation_pred_all = convert_xgboost_predictions_to_eur(
    validation_pred_model_scale_all,
    selected_xgboost_config["target_mode"],
    y_train_reference=y_train,
)

In [279]:
validation_pred_all

array([180.93927002, 419.17285156, 142.98661804, ..., 216.17285156,
       235.12261963, 234.99909973], shape=(1689,))

## 26. Preview Recommended-Feature Predictions

These cells only preview the validation predictions in euro scale for the recommended-feature model. Formal model-comparison metrics and grouped diagnostics are reported in the later validation sections.

In [280]:
# Pair each validation observation with its prediction in euro scale.
recommended_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_all,
})

recommended_prediction_preview.head()

,actual_price,predicted_price
0,177.6,180.939270
1,473.6,419.172852
2,142.1,142.986618
3,82.9,92.516640
4,177.6,122.387810


In [281]:
# A quick descriptive summary helps check the prediction range before detailed validation.
recommended_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,264.591754
std,567.153248,567.897966
min,5.900000,9.978323
25%,59.000000,57.050159
50%,100.600000,105.493568
75%,236.000000,234.990311
max,4284.000000,4297.957520


## 27. Validation Error Analysis By Part Group

This section evaluates validation-set errors for the existing XGBoost model by part grouping. It reuses the available validation predictions and reports which groups are predicted most accurately and least accurately.


In [282]:
# Reuse the most detailed validation predictions already available in the notebook.
if "best_validation_predictions" in globals():
    validation_predictions_eur = best_validation_predictions
elif "validation_pred_all" in globals():
    validation_predictions_eur = validation_pred_all
elif "validation_pred_traficom" in globals():
    validation_predictions_eur = validation_pred_traficom
elif "validation_pred" in globals():
    validation_predictions_eur = validation_pred
else:
    raise NameError("No validation predictions were found in the notebook.")

# Build one validation results table in euro scale for grouped error analysis.
validation_results_df = validation_df[["price", "category", "subcategory", "part_name"]].copy()
validation_results_df = validation_results_df.rename(columns={"price": "actual_price"})
validation_results_df["predicted_price"] = validation_predictions_eur
validation_results_df["absolute_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
).abs()
validation_results_df["squared_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
) ** 2

validation_results_df.head()

,actual_price,category,subcategory,part_name,predicted_price,absolute_error,squared_error
0,177.6,airbag,contact roll airbag,"Contact roll Airbag - , e-",174.599197,3.000803,9.004816
1,473.6,airbag,passenger airbag,"Passenger airbag - , e-",433.058838,40.541162,1643.585825
2,142.1,airbag,left,"Curtain airbags - , e-(Left)",147.554443,5.454443,29.750952
3,82.9,airbag,seat assembled,"Seat assembled - , e-(Right front)",85.761467,2.861467,8.187993
4,177.6,airbag,all,"Side airbags - , e-(Right)",127.212456,50.387544,2538.904616


In [283]:
def summarize_group_errors(validation_results_df, group_column, min_count=20):
    summary = (
        validation_results_df
        .groupby(group_column, observed=False)
        .agg(
            count=("actual_price", "size"),
            MAE=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
            within_10_eur=("absolute_error", lambda s: (s <= 10).mean()),
            within_20_eur=("absolute_error", lambda s: (s <= 20).mean()),
            within_50_eur=("absolute_error", lambda s: (s <= 50).mean()),
        )
        .reset_index()
    )

    summary = summary[summary["count"] >= min_count].copy()
    return summary.sort_values(["MAE", "RMSE", "count"]).reset_index(drop=True)


category_error_summary = summarize_group_errors(validation_results_df, "category")
subcategory_error_summary = summarize_group_errors(validation_results_df, "subcategory")
part_name_error_summary = summarize_group_errors(validation_results_df, "part_name")

## 28. Best And Worst Groups By MAE

The tables below summarize grouped validation errors in absolute euro terms. The `within_10_eur`, `within_20_eur`, and `within_50_eur` columns show how often predictions fall within practical absolute-error thresholds.

In [284]:
# Show the main absolute-error metrics used for grouped validation review.
summary_display_columns = [
    "count",
    "MAE",
    "median_absolute_error",
    "RMSE",
    "within_10_eur",
    "within_20_eur",
    "within_50_eur",
]

for summary_name, summary_df, group_label in [
    ("Category", category_error_summary, "category"),
    ("Subcategory", subcategory_error_summary, "subcategory"),
    ("Part name", part_name_error_summary, "part_name"),
]:
    print(f"\n{summary_name}: 10 best groups by MAE")
    display(
        summary_df.head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )

    print(f"{summary_name}: 10 worst groups by MAE")
    display(
        summary_df.sort_values(["MAE", "RMSE", "count"], ascending=[False, False, False]).head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )



Category: 10 best groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,fuel,181,9.36,7.04,12.90,64.6%,86.2%,100.0%
1,airbag,106,17.99,12.30,25.46,43.4%,67.0%,88.7%
2,electric / transmitter / databox / sensor,404,18.41,10.08,38.76,49.3%,80.0%,90.3%
3,vehicle exterior / suspension,294,24.18,10.77,41.47,45.6%,69.0%,85.4%
4,brakes,214,27.39,9.76,54.69,50.9%,70.6%,83.6%
5,gear box / drive axle / middle axle,241,32.85,16.87,55.61,40.7%,52.7%,75.9%
6,engine,249,39.26,13.57,83.47,43.0%,60.2%,77.5%


Category: 10 worst groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
6,engine,249,39.26,13.57,83.47,43.0%,60.2%,77.5%
5,gear box / drive axle / middle axle,241,32.85,16.87,55.61,40.7%,52.7%,75.9%
4,brakes,214,27.39,9.76,54.69,50.9%,70.6%,83.6%
3,vehicle exterior / suspension,294,24.18,10.77,41.47,45.6%,69.0%,85.4%
2,electric / transmitter / databox / sensor,404,18.41,10.08,38.76,49.3%,80.0%,90.3%
1,airbag,106,17.99,12.30,25.46,43.4%,67.0%,88.7%
0,fuel,181,9.36,7.04,12.90,64.6%,86.2%,100.0%



Subcategory: 10 best groups by MAE


...

,subcategory,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
23,engine gasoline,21,141.46,134.78,189.77,23.8%,28.6%,38.1%
22,right rear,49,46.66,50.31,58.86,26.5%,38.8%,49.0%
21,adaptiv farthållare sensor,24,36.49,22.91,44.22,0.0%,16.7%,75.0%
20,motor cushion,25,36.46,9.72,59.25,52.0%,72.0%,76.0%
19,left rear,50,29.55,16.97,46.72,28.0%,52.0%,90.0%
18,passenger airbag,25,29.24,23.71,34.15,12.0%,36.0%,84.0%
17,airbag control unit,20,25.73,11.39,31.92,25.0%,55.0%,75.0%
16,rear,43,22.35,11.47,36.68,41.9%,69.8%,86.0%
15,all,153,17.59,8.22,29.12,56.9%,74.5%,88.9%
14,right front,48,15.65,7.93,24.38,60.4%,79.2%,87.5%



Part name: 10 best groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,Distributors Vacuum regulator -,20,1.82,1.50,2.14,100.0%,100.0%,100.0%
1,ABS Hydraulic pump -,24,2.89,1.96,3.87,95.8%,100.0%,100.0%
2,Trailing link rear -(Left),20,7.31,5.54,10.83,90.0%,95.0%,100.0%
3,Brake Caliper -(Left front),20,9.70,9.13,13.17,65.0%,95.0%,100.0%
4,Drive shaft -(Left front),32,17.30,26.12,20.98,43.8%,46.9%,100.0%
5,Wheel bearing spindle shaft -(Right rear),20,27.00,10.39,36.39,40.0%,60.0%,90.0%
6,Drive shaft -(Right front),23,27.02,19.97,34.81,30.4%,52.2%,73.9%
7,Wheel bearing spindle shaft -(Left rear),21,27.55,16.38,40.08,33.3%,61.9%,95.2%


Part name: 10 worst groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
7,Wheel bearing spindle shaft -(Left rear),21,27.55,16.38,40.08,33.3%,61.9%,95.2%
6,Drive shaft -(Right front),23,27.02,19.97,34.81,30.4%,52.2%,73.9%
5,Wheel bearing spindle shaft -(Right rear),20,27.00,10.39,36.39,40.0%,60.0%,90.0%
4,Drive shaft -(Left front),32,17.30,26.12,20.98,43.8%,46.9%,100.0%
3,Brake Caliper -(Left front),20,9.70,9.13,13.17,65.0%,95.0%,100.0%
2,Trailing link rear -(Left),20,7.31,5.54,10.83,90.0%,95.0%,100.0%
1,ABS Hydraulic pump -,24,2.89,1.96,3.87,95.8%,100.0%,100.0%
0,Distributors Vacuum regulator -,20,1.82,1.50,2.14,100.0%,100.0%,100.0%


## 29. Select The Best XGBoost Feature Set

Trusted model selection is limited to leakage-safe feature sets. Exploratory feature sets that include full-history listing features are still reported, but they are excluded from automatic best-model selection.

In [ ]:
feature_sets = {
    "baseline only": {
        "features": features_baseline_only,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core": {
        "features": features_baseline_plus_traficom,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core + registry_lifecycle": {
        "features": features_baseline_plus_traficom + registry_lifecycle_features,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core + registry_lifecycle without oem_number": {
        "features": [
            column for column in (features_baseline_plus_traficom + registry_lifecycle_features)
            if column != "oem_number"
        ],
        "trusted_for_selection": True,
    },
    "baseline + traficom_core + registry_lifecycle + traficom_extended": {
        "features": (
            features_baseline_plus_traficom
            + registry_lifecycle_features
            + traficom_extended_features
        ),
        "trusted_for_selection": True,
    },
    "trusted manual all-features usable by XGBoost": {
        "features": manual_all_xgboost_features_leakage_safe,
        "trusted_for_selection": True,
    },
    "trusted recommended features": {
        "features": recommended_model_features_leakage_safe,
        "trusted_for_selection": True,
    },
    "trusted recommended features without date offsets": {
        "features": recommended_model_features_leakage_safe_without_date_offsets,
        "trusted_for_selection": True,
    },
    "trusted recommended features without date offsets without oem_number": {
        "features": [
            column for column in recommended_model_features_leakage_safe_without_date_offsets
            if column != "oem_number"
        ],
        "trusted_for_selection": True,
    },
}

experiment_results = []
experiment_feature_details = []
experiment_predictions = {}


In [286]:
def prepare_experiment_features(feature_list, train_df, validation_df, excluded_features):
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [
        feature for feature in requested_features
        if feature not in excluded_features
    ]

    available_features = [
        feature for feature in requested_features
        if feature in train_df.columns
    ]
    missing_features = [
        feature for feature in requested_features
        if feature not in train_df.columns
    ]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    return (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    )


In [287]:
for model_name, feature_config in feature_sets.items():
    feature_list = feature_config["features"]
    trusted_for_selection = feature_config["trusted_for_selection"]

    (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    ) = prepare_experiment_features(
        feature_list,
        train_df,
        validation_df,
        xgboost_excluded_features,
    )

    model_current, X_train_current_prepared, X_validation_current_prepared = fit_xgboost_model(
        X_train_current,
        prepare_xgboost_target(y_train, selected_xgboost_config["target_mode"]),
        X_validation_current,
        prepare_xgboost_target(y_validation, selected_xgboost_config["target_mode"]),
        selected_xgboost_params,
    )

    validation_pred_model_scale_current = model_current.predict(X_validation_current_prepared)
    validation_pred_current = convert_xgboost_predictions_to_eur(
        validation_pred_model_scale_current,
        selected_xgboost_config["target_mode"],
        y_train_reference=y_train,
    )
    experiment_predictions[model_name] = validation_pred_current

    validation_mae_current = mean_absolute_error(
        y_validation,
        validation_pred_current,
    )
    validation_rmse_current = np.sqrt(
        mean_squared_error(y_validation, validation_pred_current)
    )
    validation_r2_current = r2_score(
        y_validation,
        validation_pred_current,
    )

    experiment_results.append({
        "experiment": model_name,
        "trusted_for_selection": trusted_for_selection,
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
        "validation_R2": validation_r2_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "trusted_for_selection": trusted_for_selection,
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })

## 30. Validate The Best XGBoost Model
 MAE as the main validation metrics, backup R^2 and MSE, and extra MAPE. 


In [288]:
# Rebuild the best-model validation dataframe if this section is run on its own.
if "best_model_validation_df" not in globals():
    if "experiment_results" not in globals() or len(experiment_results) == 0:
        raise NameError(
            "experiment_results is not defined. Run the feature-set comparison section first."
        )

    experiment_results_df = pd.DataFrame(experiment_results)
    if "trusted_for_selection" not in experiment_results_df.columns:
        experiment_results_df["trusted_for_selection"] = True

    trusted_experiment_results_df = experiment_results_df[
        experiment_results_df["trusted_for_selection"]
    ].copy()
    if trusted_experiment_results_df.empty:
        trusted_experiment_results_df = experiment_results_df.copy()

    best_experiment_name = trusted_experiment_results_df.sort_values(
        ["validation_MAE", "validation_RMSE"]
    ).iloc[0]["experiment"]

    if "experiment_predictions" not in globals() or best_experiment_name not in experiment_predictions:
        raise NameError(
            "experiment_predictions is missing the current best experiment. Rerun the feature-set comparison section."
        )

    best_validation_predictions = experiment_predictions[best_experiment_name]

    best_model_validation_df = pd.DataFrame({
        "actual_price": y_validation,
        "predicted_price": best_validation_predictions,
    })
    best_model_validation_df["residual"] = (
        best_model_validation_df["actual_price"]
        - best_model_validation_df["predicted_price"]
    )
    best_model_validation_df["absolute_error"] = (
        best_model_validation_df["residual"].abs()
    )
    best_model_validation_df["squared_error"] = (
        best_model_validation_df["residual"] ** 2
    )
    best_model_validation_df["absolute_percentage_error"] = (
        best_model_validation_df["absolute_error"]
        / best_model_validation_df["actual_price"].clip(lower=1)
    )
    best_model_validation_df["prediction_direction"] = np.where(
        best_model_validation_df["residual"] >= 0,
        "underpredicted",
        "overpredicted",
    )
    best_model_validation_df["actual_price_band"] = pd.qcut(
        best_model_validation_df["actual_price"],
        q=5,
        duplicates="drop",
    )

# Split the best-model validation errors by price band and prediction direction.
best_model_direction_summary = (
    best_model_validation_df
    .groupby(["actual_price_band", "prediction_direction"], observed=False)
    .agg(
        samples=("actual_price", "size"),
        mean_actual_price=("actual_price", "mean"),
        mean_predicted_price=("predicted_price", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

best_model_direction_summary.style.format({
    "mean_actual_price": "{:.1f}",
    "mean_predicted_price": "{:.1f}",
    "mae": "{:.2f}",
})


,actual_price_band,prediction_direction,samples,mean_actual_price,mean_predicted_price,mae
0,"(5.899, 47.4]",overpredicted,180,35.4,52.7,17.28
1,"(5.899, 47.4]",underpredicted,169,35.6,28.6,7.02
2,"(47.4, 82.6]",overpredicted,216,60.2,75.2,14.96
3,"(47.4, 82.6]",underpredicted,119,62.8,52.5,10.36
4,"(82.6, 154.06]",overpredicted,169,113.0,125.3,12.28
5,"(82.6, 154.06]",underpredicted,160,106.7,91.2,15.47
6,"(154.06, 248.42]",overpredicted,158,202.7,243.7,41.03
7,"(154.06, 248.42]",underpredicted,180,207.3,187.2,20.08
8,"(248.42, 4284.0]",overpredicted,105,1090.1,1160.5,70.50
9,"(248.42, 4284.0]",underpredicted,233,790.9,744.2,46.75


In [289]:
# Rebuild the best-model validation dataframe from the current experiment results.
if "experiment_results" not in globals() or len(experiment_results) == 0:
    raise NameError(
        "experiment_results is not defined. Run the feature-set comparison section first."
    )

experiment_results_df = pd.DataFrame(experiment_results)
if "trusted_for_selection" not in experiment_results_df.columns:
    experiment_results_df["trusted_for_selection"] = True

trusted_experiment_results_df = experiment_results_df[
    experiment_results_df["trusted_for_selection"]
].copy()
if trusted_experiment_results_df.empty:
    trusted_experiment_results_df = experiment_results_df.copy()

best_experiment_name = trusted_experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]

if "experiment_predictions" not in globals() or best_experiment_name not in experiment_predictions:
    raise NameError(
        "experiment_predictions is missing the current best experiment. Rerun the feature-set comparison section."
    )

best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

best_model_validation_df


,actual_price,predicted_price,residual,absolute_error,squared_error,absolute_percentage_error,prediction_direction,actual_price_band
0,177.6,177.180313,0.419687,0.419687,0.176137,0.002363,underpredicted,"(154.06, 248.42]"
1,473.6,439.963196,33.636804,33.636804,1131.434597,0.071024,underpredicted,"(248.42, 4284.0]"
2,142.1,144.092758,-1.992758,1.992758,3.971085,0.014024,overpredicted,"(82.6, 154.06]"
3,82.9,94.899963,-11.999963,11.999963,143.999121,0.144752,overpredicted,"(82.6, 154.06]"
4,177.6,118.893845,58.706155,58.706155,3446.412681,0.330553,underpredicted,"(154.06, 248.42]"
...,...,...,...,...,...,...,...,...
1684,212.9,211.824509,1.075491,1.075491,1.156682,0.005052,underpredicted,"(154.06, 248.42]"
1685,212.9,208.381729,4.518271,4.518271,20.414772,0.021223,underpredicted,"(154.06, 248.42]"
1686,212.9,217.170090,-4.270090,4.270090,18.233666,0.020057,overpredicted,"(154.06, 248.42]"
1687,236.5,234.409683,2.090317,2.090317,4.369424,0.008839,underpredicted,"(154.06, 248.42]"


In [290]:
experiment_results_df = pd.DataFrame(experiment_results)
if "trusted_for_selection" not in experiment_results_df.columns:
    experiment_results_df["trusted_for_selection"] = True

trusted_experiment_results_df = experiment_results_df[
    experiment_results_df["trusted_for_selection"]
].copy()
if trusted_experiment_results_df.empty:
    trusted_experiment_results_df = experiment_results_df.copy()

baseline_candidates_df = trusted_experiment_results_df.loc[
    trusted_experiment_results_df["experiment"] == "baseline only"
]
if baseline_candidates_df.empty:
    baseline_candidates_df = experiment_results_df.loc[
        experiment_results_df["experiment"] == "baseline only"
    ]
if baseline_candidates_df.empty:
    raise NameError("Could not find the 'baseline only' row in experiment_results.")

baseline_row = baseline_candidates_df.iloc[0]

experiment_results_df["delta_MAE_vs_baseline"] = (
    experiment_results_df["validation_MAE"]
    - baseline_row["validation_MAE"]
)
experiment_results_df["delta_RMSE_vs_baseline"] = (
    experiment_results_df["validation_RMSE"]
    - baseline_row["validation_RMSE"]
)
experiment_results_df["mae_rank"] = (
    experiment_results_df["validation_MAE"]
    .rank(method="dense")
    .astype(int)
)
experiment_results_df["rmse_rank"] = (
    experiment_results_df["validation_RMSE"]
    .rank(method="dense")
    .astype(int)
)

best_experiment_name = trusted_experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]

display_columns = [
    "experiment",
    "trusted_for_selection",
    "raw_columns_used",
    "validation_MAE",
    "delta_MAE_vs_baseline",
    "mae_rank",
    "validation_RMSE",
    "delta_RMSE_vs_baseline",
    "rmse_rank",
    "validation_R2",
]

experiment_results_df.sort_values(
    ["trusted_for_selection", "validation_MAE", "validation_RMSE"],
    ascending=[False, True, True],
)[display_columns].style.format({
    "validation_MAE": "{:.4f}",
    "delta_MAE_vs_baseline": "{:+.4f}",
    "validation_RMSE": "{:.4f}",
    "delta_RMSE_vs_baseline": "{:+.4f}",
    "validation_R2": "{:.4f}",
})


,experiment,trusted_for_selection,raw_columns_used,validation_MAE,delta_MAE_vs_baseline,mae_rank,validation_RMSE,delta_RMSE_vs_baseline,rmse_rank,validation_R2
8,trusted recommended features without date offsets without oem_number,True,65,21.6889,-7.3510,3,52.6047,-23.5633,3,0.9914
6,trusted recommended features,True,67,24.3555,-4.6844,4,60.4940,-15.6740,5,0.9886
3,baseline + traficom_core + registry_lifecycle without oem_number,True,40,24.5168,-4.5231,5,61.5463,-14.6217,6,0.9882
7,trusted recommended features without date offsets,True,66,24.5908,-4.4491,6,58.7190,-17.4491,4,0.9893
5,trusted manual all-features usable by XGBoost,True,63,26.7559,-2.2839,7,67.6405,-8.5275,7,0.9858
4,baseline + traficom_core + registry_lifecycle + traficom_extended,True,63,26.8747,-2.1652,8,68.5238,-7.6442,8,0.9854
2,baseline + traficom_core + registry_lifecycle,True,41,28.3293,-0.7105,9,72.8767,-3.2913,9,0.9835
0,baseline only,True,14,29.0399,+0.0000,10,76.1680,+0.0000,10,0.9820
1,baseline + traficom_core,True,27,29.9531,+0.9132,11,88.2047,+12.0367,11,0.9758
10,exploratory all recommended features without date offsets,False,76,14.9749,-14.0650,1,40.4191,-35.7489,1,0.9949
